# Exercice 04 – Extraire les données de contrats de leasing

Le dossier `contrats/` contient 20 contrats de leasing automobile (fictifs), en PDF, issus de trois agences différentes. Chaque agence a sa propre mise en page et son propre vocabulaire (*client* ou *preneur*, *mensualité*, *loyer* ou *redevance*…). Certains contrats font deux pages.

**Objectif** : pour chaque contrat, extraire les informations ci-dessous avec un modèle vision et une **sortie structurée** (schéma Pydantic), les rassembler dans un `DataFrame` **polars**, et mesurer la qualité de l'extraction.

**Serveur** : un modèle vision avec son `mmproj` (voir le cours sur les images). Pensez à vérifier `requests.get(f"{BASE_URL}/props").json()["modalities"]`.

## Les informations à extraire

| Champ | Type | Remarque |
|---|---|---|
| `numero_contrat` | texte | ex. `ALW-2026-1914` |
| `date_contrat` | date | au format `AAAA-MM-JJ` quel que soit le format sur le document |
| `agence` | texte | nom de la société de leasing |
| `client` | texte | le client (ou « preneur »), **pas** le conseiller commercial |
| `vehicule.marque`, `vehicule.modele` | texte | |
| `vehicule.carburant` | choix | `essence`, `diesel`, `hybride`, `hybride rechargeable`, `électrique` |
| `duree_mois`, `km_annuel` | entier | |
| `acompte_eur` | nombre | 0 s'il n'y en a pas |
| `mensualite_tvac_eur` | nombre | TVA comprise ; si le montant a été corrigé à la main, la valeur corrigée |
| `assurance.type` | choix | `omnium`, `mini-omnium`, `rc`, `aucune` |
| `assurance.compagnie` | texte | vide si RC seule ou aucune assurance |
| `assurance.prime_mensuelle_eur` | nombre | 0 si aucune prime séparée |
| `signature_client`, `signature_agence` | booléen | `True` seulement si une signature manuscrite est présente |
| `annule` | booléen | `True` si le document porte une mention d'annulation |

## Ouvrir un PDF en Python

On utilise `pymupdf` (`pip install pymupdf`). Un PDF peut contenir :

- **du texte** (PDF « natif », généré par un logiciel) : `page.get_text()` le renvoie directement ;
- **seulement des images** (PDF **scanné**) : `page.get_text()` renvoie une chaîne vide, il faut transformer la page en image et la donner au modèle vision.

Dans ce lot, **certains contrats sont natifs, d'autres sont des scans**. Le code suivant vous montre la différence.

## À faire

1. Écrire les classes Pydantic correspondant au tableau des champs (`Vehicule`, `Assurance`, `Contrat`).
2. Écrire `extraire(chemin_pdf) -> Contrat` qui envoie **toutes les pages** du contrat au modèle, avec `response_format` et le schéma.
3. Boucler sur les 20 contrats, enregistrer les résultats (plus `prompt_tokens` et la durée) dans `resultats_contrats.csv`.
4. Évaluer : le fichier `verite_terrain.json` avec les valeurs attendues vous sera donné en fin de séance. Calculez le taux de bonnes réponses **par champ**, puis séparément pour les contrats **natifs** et **scannés**.
5. Répondez : quels champs posent le plus de problèmes ? Pourquoi ? Qu'est-ce qui change entre un PDF natif et un scan ?

> **Tips**
> - Le system prompt compte autant que le schéma : rappelez les règles du tableau (TVAC, corrections manuscrites, client ≠ conseiller, ce qui compte comme signature).
> - Les `description` de `Field(...)` sont envoyées au modèle avec le schéma : utilisez-les.
> - Une page A4 à 110 dpi coûte environ un millier de tokens : vérifiez que votre `-c` suffit pour un contrat de deux pages.
> - `temperature=0` et vérifiez `finish_reason`.
> - `pl.DataFrame(liste_de_dicts)` puis `.write_csv(...)` ; `pl.read_json("verite_terrain.json")` lit la vérité terrain.
> - Pour les PDF natifs, vous pouvez **aussi** envoyer le texte extrait. Est-ce que ça aide ? Toujours ?

In [43]:
from pydantic import BaseModel, Field
from typing import Literal
import glob
import os
import time
import polars as pl
from datetime import date

class Vehicle(BaseModel):
  make: str = Field(..., description="The make of the vehicle")
  model: str = Field(..., description="The model of the vehicle")
  motor: Literal['essence', 'diesel', 'hybride', 'hybride rechargeable', 'électrique'] = Field(..., description="The fuel / motor type of the vehicle")

class Insurance(BaseModel):
  type: Literal['omnium', 'mini-omnium', 'rc', 'aucune'] = Field(..., description="The type of insurance; 'aucune' if there is no insurance")
  company: str = Field(..., description="The insurance company; empty string if insurance is 'rc' only or 'aucune'")
  monthly_prime: float = Field(..., description="The separate monthly insurance premium, in euros; 0 if there is no separate premium")

class Contract(BaseModel):
  contract_number: str = Field(..., description="The contract number, e.g. ALW-2026-1914")
  contract_date: date = Field(..., description="The date of the contract. Format YYYY-MM-DD, whatever the format on the document")
  insurance: Insurance = Field(..., description="The insurance covering the leasing contract")
  vehicle: Vehicle = Field(..., description="The vehicle being leased")
  agency: str = Field(..., description="The name of the leasing company")
  client: str = Field(..., description="The name of the customer (or 'preneur'), NOT the sales advisor")
  duration: int = Field(..., description="Duration of the contract, in months")
  annual_km: int = Field(..., description="The maximum amount of kilometers per year that the contract allows")
  deposit: float = Field(..., description="Deposit amount, in euros; 0 if there is none")
  monthly_amount: float = Field(..., description="Monthly payment amount, VAT included (TVAC); if the amount was corrected by hand, the corrected amount")
  client_signature: bool = Field(..., description="True only if a handwritten signature of the client is present")
  agency_signature: bool = Field(..., description="True only if a handwritten signature of the agency is present")
  cancel: bool = Field(..., description="True if the document bears a cancellation mark (e.g. 'ANNULÉ' stamp), not just a clause explaining how to cancel")

In [44]:
from openai import OpenAI

BASE_URL = '127.0.0.1:9931'
client = OpenAI(base_url=f'http://{BASE_URL}/v1', api_key='NOKEY')


In [45]:
import base64
import pymupdf
from IPython.display import Image, display


In [ ]:
def read_pdf(pdf_path):
  doc = pymupdf.open(pdf_path)
  text_data = ""
  images_data = []
  for page in doc:
    text_data += page.get_text()

    pix = page.get_pixmap(dpi=110)
    png = pix.tobytes("png")
    display(Image(png, width=450))

    encoded_png = base64.b64encode(png).decode()
    images_data.append(f"data:image/png;base64,{encoded_png}")
  return text_data, images_data

In [ ]:
SYSTEM_PROMPT = """Extract the data of this car leasing contract into the JSON schema.
Rules:
- The client (or "preneur") is the customer, NOT the sales advisor.
- The monthly amount is VAT included (TVAC). If an amount was corrected by hand, use the corrected value.
- Signatures are True only if a handwritten signature is present (a printed name or empty box is not a signature).
- cancel is True only if the document bears a cancellation mark.
- Dates in YYYY-MM-DD format."""

def request_model(text_data, images_data):
  content = [{"type": "image_url", "image_url": {"url": image}} for image in images_data]
  
  if text_data.strip():
    content.append({"type": "text", "text": f"<document>\n{text_data}\n</document>"})

  r = client.chat.completions.create(
    model='local',
    messages=[
      {'role': 'system', 'content': SYSTEM_PROMPT},
      {'role': 'user', 'content': content}
    ],
    temperature=0,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    response_format={
      "type": "json_schema",
      "json_schema": {
        "name": "leasing_contract",
        "schema": Contract.model_json_schema()
      }
    }
  )
  if r.choices[0].finish_reason != "stop":
    print(f"Warning: finish_reason = {r.choices[0].finish_reason}")
  timings = r.model_extra['timings']
  contract = Contract.model_validate_json(r.choices[0].message.content)
  return contract, r.usage.prompt_tokens, timings

In [48]:
def extract(pdf_path) -> tuple[Contract, int]:
  text_data, images_data = read_pdf(pdf_path)
  return request_model(text_data, images_data)

In [49]:
folder = "04_contrats_leasing/contrats"
pdf_files = glob.glob(os.path.join(folder, "*.pdf"))

In [ ]:
rows = []
for path in pdf_files:
  start = time.perf_counter()
  contract, prompt_tokens, timings = extract(path)
  duration = time.perf_counter() - start
  print(contract)
  rows.append({
    "file": os.path.basename(path),
    **contract.model_dump(mode="json"),
    "prompt_tokens": prompt_tokens,
    "time": round(duration, 2),
  })

df = pl.json_normalize(rows)
df.write_csv("resultats_contrats.csv")


## Comparaison avec la vérité terrain

In [53]:
# your column -> ground-truth column
FIELD_MAPPING = {
  "contract_number": "numero_contrat",
  "contract_date": "date_contrat",
  "agency": "ville_agence",
  "client": "client",
  "duration": "duree_mois",
  "annual_km": "km_annuel",
  "deposit": "acompte_eur",
  "monthly_amount": "mensualite_tvac_eur",
  "client_signature": "signature_client",
  "agency_signature": "signature_agence",
  "cancel": "annule",
  "insurance.type": "assurance",
  "insurance.company": "assurance_compagnie",
  "insurance.monthly_prime": "assurance_prime_mensuelle_eur",
  "vehicle.make": "marque",
  "vehicle.model": "modele",
  "vehicle.motor": "carburant",
}

def normalize(value):
  if value is None:
    return None
  if isinstance(value, bool):
    return value
  if isinstance(value, (int, float)):
    return round(float(value), 2)
  s = str(value).strip().lower()
  if s == "":
    return None
  if s in ("true", "false"):
    return s == "true"
  try:
    return round(float(s.replace(",", ".")), 2)
  except ValueError:
    return s

def compare_results(df, truth_path="04_contrats_leasing/verite_terrain.json", mapping=FIELD_MAPPING):
  truth = pl.read_json(truth_path)
  truth = truth.rename({c: f"gt_{c}" for c in truth.columns})
  merged = df.join(truth, left_on="file", right_on="gt_fichier", how="inner")
  print(f"{merged.height} contracts compared\n")

  scores = []
  errors = {}
  for col, gt in mapping.items():
    preds = merged[col].to_list()
    truths = merged[f"gt_{gt}"].to_list()
    ok = [normalize(p) == normalize(t) for p, t in zip(preds, truths)]
    scores.append({"field": col, "accuracy": sum(ok) / len(ok)})
    bad = [
      {"file": f, "predicted": p, "expected": t}
      for f, p, t, good in zip(merged["file"].to_list(), preds, truths, ok) if not good
    ]
    if bad:
      errors[col] = pl.DataFrame(bad, schema={"file": pl.String, "predicted": pl.String, "expected": pl.String}, strict=False)

  scores = pl.DataFrame(scores).sort("accuracy")
  global_accuracy = scores["accuracy"].mean()

  with pl.Config(tbl_rows=-1, fmt_str_lengths=60):
    print(scores.with_columns((pl.col("accuracy") * 100).round(1).alias("accuracy %")).drop("accuracy"))
    print(f"\nGlobal accuracy: {global_accuracy:.1%}")
    for col, bad in errors.items():
      print(f"\n--- {col} ({bad.height} errors) ---")
      print(bad)

  return scores, errors

In [54]:
scores, errors = compare_results(pl.read_csv("resultats_contrats.csv"))

20 contracts compared

shape: (17, 2)
┌─────────────────────────┬────────────┐
│ field                   ┆ accuracy % │
│ ---                     ┆ ---        │
│ str                     ┆ f64        │
╞═════════════════════════╪════════════╡
│ agency                  ┆ 60.0       │
│ insurance.company       ┆ 80.0       │
│ client                  ┆ 95.0       │
│ client_signature        ┆ 95.0       │
│ insurance.type          ┆ 95.0       │
│ vehicle.model           ┆ 95.0       │
│ contract_number         ┆ 100.0      │
│ contract_date           ┆ 100.0      │
│ duration                ┆ 100.0      │
│ annual_km               ┆ 100.0      │
│ deposit                 ┆ 100.0      │
│ monthly_amount          ┆ 100.0      │
│ agency_signature        ┆ 100.0      │
│ cancel                  ┆ 100.0      │
│ insurance.monthly_prime ┆ 100.0      │
│ vehicle.make            ┆ 100.0      │
│ vehicle.motor           ┆ 100.0      │
└─────────────────────────┴────────────┘

Global accuracy: 9